# Phase 58 — Thread 1 Redone at Round-3 Scale: Does More Data Resolve the Diffuse Topology Signal?

At the original pool scale (2,056 clusters), the proper 8-seed matched
comparison (Phase26) found H0/H1/H2/burial statistically indistinguishable
from each other -- every pairwise gap sat under ~1x the seed noise floor,
nowhere near the 3x+ threshold that made the topology-vs-no-topology
finding (Phase22) credible. **Standing conclusion at that scale: the
topology benefit is diffuse across all four pieces, not concentrated in
one.**

This redoes the test at round-3 scale (7,513 clusters) using the
leakage-clean-training-against-COACH420 methodology Phase53/56 already
validated -- necessary because Thread 1's original benchmark
(`phase15_raw_features.pkl`) was only leakage-verified against the
ORIGINAL 2,056-chain pool, and that guarantee does not automatically
extend to the much larger round-3 training pool. Phase56's already-trained
`full` arm (COACH420-clean, round-3 pool) is reused directly -- only the
five other arms (no_topo, H0/H1/H2/burial-only) need fresh training, at a
MATCHED 3 seeds each from the start, avoiding the ensemble-size-mismatch
confound that undermined the original Phase23.

**Question:** does more training data sharpen the topology-decomposition
signal into something attributable, or does the diffuse-signal finding
hold at scale too?

## Cell 1 — Setup (loads Phase 52/51's round-3 scaled blobs, identical to Phase53/56)

In [ ]:
!pip install -q torch-geometric==2.6.1 gudhi==3.10.1 biopython==1.84 transformers==4.46.3
!apt-get -qq update > /dev/null 2>&1 && apt-get -qq install -y default-jre-headless > /dev/null 2>&1

In [ ]:
import csv as _csv
import gzip, json, os, pickle, shutil, subprocess, time, urllib.request, warnings
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from scipy.spatial import cKDTree
from torch_geometric.loader import DataLoader
from torch_geometric.utils import softmax as pyg_softmax
from sklearn.metrics import average_precision_score, matthews_corrcoef, roc_auc_score

warnings.filterwarnings("ignore")

def locate(f):
    c=[]
    for r in (Path("/kaggle/input"), Path("/kaggle/working"), Path(".")):
        if r.exists(): c.extend(r.rglob(f))
    if not c: raise FileNotFoundError(f)
    return max(c, key=lambda p: p.stat().st_size)

blob10 = torch.load(locate("phase52_graphs.pt"), map_location="cpu", weights_only=False)
blob6b = torch.load(locate("phase51_graphs.pt"), map_location="cpu", weights_only=False)
dataset = blob10["dataset"]
N_PRE = int(blob10["n_pre_topo"]); TOPO_DIM = int(blob10["topo_dim"])
POS_WEIGHT = float(blob10["pos_weight"])
TOPO_NAMES = blob6b["topo_names"]

GPU_OK=False
if torch.cuda.is_available():
    try:
        _p=torch.randn(256,256,device="cuda"); (_p@_p).sum().item(); torch.cuda.synchronize(); GPU_OK=True
        print(f"GPU: {torch.cuda.get_device_name(0)} (probe PASS)")
    except RuntimeError as e: print("GPU unusable:", e)
DEVICE = torch.device("cuda" if GPU_OK else "cpu")
print(f"training set: {len(dataset):,} chains")

WORK = Path("/kaggle/working"); (WORK/"cache").mkdir(parents=True, exist_ok=True)
RAW = Path("/tmp/pdb_raw"); RAW.mkdir(parents=True, exist_ok=True)

## Cell 2 — Fetch ALL 420 COACH420 structures (identical to Phase 53/56)

In [ ]:
def parse_ds(url):
    lines = urllib.request.urlopen(url, timeout=60).read().decode().splitlines()
    out = []
    for ln in lines:
        ln = ln.strip()
        if not ln or ln.startswith("#"): continue
        name = ln.split("/")[-1].replace(".pdb", "")
        out.append((name[:4].upper(), name[4:].upper() or None))
    return out

coach = parse_ds("https://raw.githubusercontent.com/rdk/p2rank-datasets/master/coach420.ds")
print(f"COACH420: {len(coach)} entries (full benchmark, no filtering)")

need = sorted({pid for pid, _ in coach})
def fetch(pid):
    for ext in ("pdb","cif"):
        p = RAW / f"{pid}.{ext}.gz"
        if p.exists() and p.stat().st_size > 500: return pid, p
    for ext in ("pdb","cif"):
        try:
            with urllib.request.urlopen(f"https://files.rcsb.org/download/{pid}.{ext}.gz", timeout=60) as r:
                b = r.read()
            if len(b) > 500: p = RAW/f"{pid}.{ext}.gz"; p.write_bytes(b); return pid, p
        except Exception: continue
    return pid, None

t0=time.time()
with ThreadPoolExecutor(max_workers=24) as ex: got = list(ex.map(fetch, need))
coach_paths = {p: f for p, f in got if f}
print(f"fetched {len(coach_paths):,}/{len(need):,} in {(time.time()-t0)/60:.1f} min")

## Cell 3 — Clean the TRAINING side: exclude any chain homologous to COACH420 (identical to Phase 53/56; reproduces the same 7,181 chains)

In [ ]:
from Bio.PDB import PDBParser

AA3_TO_1={"ALA":"A","ARG":"R","ASN":"N","ASP":"D","CYS":"C","GLN":"Q","GLU":"E","GLY":"G",
          "HIS":"H","ILE":"I","LEU":"L","LYS":"K","MET":"M","PHE":"F","PRO":"P","SER":"S",
          "THR":"T","TRP":"W","TYR":"Y","VAL":"V","MSE":"M"}

def get_seq(pid, path, chain_id):
    try:
        with gzip.open(path, "rt") as fh:
            model = next(PDBParser(QUIET=True, PERMISSIVE=True).get_structure(pid, fh).get_models())
        ch = model[chain_id] if chain_id else next(iter(model))
        residues = [r for r in ch if r.get_resname().strip().upper() in AA3_TO_1 and "CA" in r]
        return "".join(AA3_TO_1.get(r.get_resname().strip().upper(),"X") for r in residues)
    except Exception:
        return None

coach_seqs = {}
for pid, ch in coach:
    if pid not in coach_paths: continue
    seq = get_seq(pid, coach_paths[pid], ch)
    if seq: coach_seqs[f"{pid}_{ch or 'A'}"] = seq
print(f"COACH420 sequences recovered: {len(coach_seqs)}/{len(coach)}")

mm = None
try:
    subprocess.run("curl -sL https://mmseqs.com/latest/mmseqs-linux-avx2.tar.gz -o /tmp/mm.tar.gz "
                   "&& tar xzf /tmp/mm.tar.gz -C /tmp", shell=True, check=True, timeout=600)
    c = Path("/tmp/mmseqs/bin/mmseqs")
    if c.exists(): os.chmod(c, 0o755); mm = str(c)
except Exception as e:
    print("mmseqs2 unavailable:", type(e).__name__)

W = Path("/tmp/rev_leak"); shutil.rmtree(W, ignore_errors=True); W.mkdir()
fasta_lines = [f">TRAIN__{d.pdb_id}_{d.chain_id}\n{d.seq}\n" for d in dataset]
fasta_lines += [f">BENCH__{k}\n{v}\n" for k, v in coach_seqs.items()]
(W/"all.fasta").write_text("".join(fasta_lines))

subprocess.run([mm, "easy-cluster", str(W/"all.fasta"), str(W/"res"), str(W/"tmp"),
                "--min-seq-id", "0.30", "-c", "0.8", "--cov-mode", "1", "-v", "1"],
               check=True, capture_output=True, timeout=3600)
rep = defaultdict(list)
for line in (W/"res_cluster.tsv").read_text().splitlines():
    a, b = line.split("\t"); rep[a].append(b)
for r_, members in rep.items():
    if r_ not in members: members.append(r_)

leaked_train_keys = set()
for members in rep.values():
    has_train = any(m.startswith("TRAIN__") for m in members)
    has_bench = any(m.startswith("BENCH__") for m in members)
    if has_train and has_bench:
        leaked_train_keys.update(m for m in members if m.startswith("TRAIN__"))

print(f"training chains homologous to COACH420 (excluded): {len(leaked_train_keys)}/{len(dataset)}")

clean_dataset = [d for d in dataset if f"TRAIN__{d.pdb_id}_{d.chain_id}" not in leaked_train_keys]
print(f"clean training set: {len(clean_dataset):,}/{len(dataset):,} chains "
      f"({100*len(clean_dataset)/len(dataset):.1f}% retained)")
assert len(clean_dataset) > 0.8 * len(dataset), "too much training data excluded -- investigate before proceeding"
dataset = clean_dataset

## Cell 4 — Recover true ligand centers for the cleaned training set (identical to Phase 53/56)

In [ ]:
from Bio.PDB import PDBParser

AA3_TO_1={"ALA":"A","ARG":"R","ASN":"N","ASP":"D","CYS":"C","GLN":"Q","GLU":"E","GLY":"G",
          "HIS":"H","ILE":"I","LEU":"L","LYS":"K","MET":"M","PHE":"F","PRO":"P","SER":"S",
          "THR":"T","TRP":"W","TYR":"Y","VAL":"V","MSE":"M"}
NON_LIGAND={"HOH","DOD","WAT","SO4","PO4","GOL","EDO","PEG","PG4","PGE","1PE","MPD","TRS",
            "ACT","ACY","FMT","CIT","MES","EPE","DMS","IMD","BME","NO3","AZI","CO3","IOD",
            "BR","CL","F","NA","K","CS","RB","LI","UNX","UNL","UNK"}
CATALYTIC_METALS={"ZN","MG","MN","FE","FE2","CU","CU1","CA","NI","CO","CD","W","MO"}
MIN_LIG_ATOMS = 6
RAW = Path("/tmp/pdb_raw"); RAW.mkdir(parents=True, exist_ok=True)

need = sorted({d.pdb_id for d in dataset})
def fetch(pid):
    for ext in ("pdb","cif"):
        p = RAW / f"{pid}.{ext}.gz"
        if p.exists() and p.stat().st_size > 500: return pid, p
    for ext in ("pdb","cif"):
        try:
            with urllib.request.urlopen(f"https://files.rcsb.org/download/{pid}.{ext}.gz", timeout=60) as r:
                b = r.read()
            if len(b) > 500: p = RAW/f"{pid}.{ext}.gz"; p.write_bytes(b); return pid, p
        except Exception: continue
    return pid, None

t0=time.time()
with ThreadPoolExecutor(max_workers=16) as ex: got = list(ex.map(fetch, need))
paths = {p: f for p, f in got if f}
print(f"fetched {len(paths):,}/{len(need):,} in {(time.time()-t0)/60:.1f} min")

def ligands_and_ca(pid, path, chain_id, n_expect):
    from Bio.PDB import MMCIFParser
    try:
        parser = MMCIFParser(QUIET=True) if ".cif" in path.name else PDBParser(QUIET=True, PERMISSIVE=True)
        with gzip.open(path,"rt") as fh:
            model = next(parser.get_structure(pid, fh).get_models())
        ch = model[chain_id]
    except Exception:
        return None
    residues=[r for r in ch if r.get_resname().strip().upper() in AA3_TO_1 and "CA" in r]
    if len(residues) != n_expect: return None
    raw_ca = np.asarray([r["CA"].coord for r in residues], dtype=np.float32)

    ligs = []
    for c2 in model:
        for res in c2:
            if res.id[0] == " ": continue
            nm = res.get_resname().strip().upper()
            if nm in AA3_TO_1 or nm in NON_LIGAND: continue
            xyz = np.asarray([a.coord for a in res.get_unpacked_list() if a.element not in ("H","D")],
                             dtype=np.float32).reshape(-1,3)
            is_metal = nm in CATALYTIC_METALS
            if is_metal or len(xyz) >= MIN_LIG_ATOMS: ligs.append(xyz)
    return raw_ca, ligs

t0=time.time(); n_ok=0; n_geom_fail=0; n_no_ligand=0
for d in dataset:
    if d.pdb_id not in paths: continue
    r = ligands_and_ca(d.pdb_id, paths[d.pdb_id], d.chain_id, int(d.num_nodes))
    if r is None: continue
    raw_ca, ligs = r
    if not ligs: n_no_ligand += 1; continue

    # geometry-drift check: pairwise CA distances must match d.pos (residue ORDER, not just count)
    k = min(50, len(raw_ca))
    a = np.linalg.norm(raw_ca[:k,None]-raw_ca[None,:k], axis=-1)
    b = np.linalg.norm(d.pos.numpy()[:k,None]-d.pos.numpy()[None,:k], axis=-1)
    if float(np.abs(a-b).max()) > 1e-2:
        n_geom_fail += 1; continue

    offset = raw_ca.mean(axis=0)                       # same centering d.pos already used
    y_np = d.y.numpy().astype(bool)
    contact_centroid = raw_ca[y_np].mean(axis=0) if y_np.any() else raw_ca.mean(axis=0)
    # "primary" ligand = the one whose center of mass is nearest the labelled contact residues
    coms = [lig.mean(axis=0) for lig in ligs]
    best = int(np.argmin([np.linalg.norm(c - contact_centroid) for c in coms]))
    d.true_center = torch.from_numpy((coms[best] - offset).astype(np.float32))
    n_ok += 1

print(f"\ntrue centers recovered: {n_ok:,}/{len(dataset):,} "
      f"({(time.time()-t0)/60:.1f} min) | geometry mismatches: {n_geom_fail} | no ligand: {n_no_ligand}")

dataset = [d for d in dataset if hasattr(d, "true_center")]
print(f"clean training chains with recovered true centers: {len(dataset):,}")

rng = np.random.RandomState(42)
perm = rng.permutation(len(dataset))
n_val = max(1, int(0.08 * len(perm)))
splits = {"val": [dataset[i] for i in perm[:n_val]], "train": [dataset[i] for i in perm[n_val:]]}
n_tr, n_va = len(splits["train"]), len(splits["val"])
print(f"deployment training: {n_tr} train / {n_va} val")

## Cell 5 — Model (identical to every prior localization phase)

In [ ]:
class EGCL(nn.Module):
    def __init__(s,h,ea=2,act=nn.SiLU,uc=True):
        super().__init__(); s.uc=uc
        s.e=nn.Sequential(nn.Linear(2*h+1+ea,h),act(),nn.Linear(h,h),act())
        s.n=nn.Sequential(nn.Linear(2*h,h),act(),nn.Linear(h,h))
        s.a=nn.Sequential(nn.Linear(h,1),nn.Sigmoid())
        if uc:
            s.c=nn.Sequential(nn.Linear(h,h),act(),nn.Linear(h,1,bias=False))
            nn.init.xavier_uniform_(s.c[-1].weight,gain=0.001)
        s.norm=nn.LayerNorm(h)
    def forward(s,h,x,ei,ea):
        src,dst=ei; rel=x[src]-x[dst]; d2=(rel**2).sum(-1,keepdim=True)
        m=s.e(torch.cat([h[src],h[dst],d2,ea],-1)); m=m*s.a(m)
        agg=torch.zeros_like(h).index_add_(0,dst,m)
        deg=torch.zeros(h.size(0),1,device=h.device).index_add_(0,dst,torch.ones_like(d2)).clamp(min=1)
        h=s.norm(h+s.n(torch.cat([h,agg/deg],-1)))
        if s.uc:
            w=s.c(m)*rel/(d2.sqrt()+1.0); x=x+torch.zeros_like(x).index_add_(0,dst,w)/deg
        return h,x

class Fus(nn.Module):
    def __init__(s,h,t):
        super().__init__(); s.p=nn.Linear(t,h); s.g=nn.Sequential(nn.Linear(h+t,h),nn.Sigmoid())
    def forward(s,h,t): return h+s.g(torch.cat([h,t],-1))*s.p(t)

class LocalizingEGNN(nn.Module):
    """Same TopologicalEGNN as every prior phase, plus a second head that
    predicts one pocket-center point per chain via a batch-aware softmax over
    a dedicated per-residue localization logit -- decoupled from the
    classification probability on purpose (see markdown above)."""
    def __init__(s, n_pre, td, h=128, L=4, dr=0.1, use_loc_head=True):
        super().__init__(); s.n_pre=n_pre; s.use_loc_head=use_loc_head
        s.embed=nn.Sequential(nn.Linear(n_pre+td,h),nn.SiLU(),nn.Linear(h,h))
        s.ls=nn.ModuleList([EGCL(h,uc=(i<L-1)) for i in range(L)])
        s.fs=nn.ModuleList([Fus(h,td) for _ in range(L)])
        s.head=nn.Sequential(nn.Linear(h,h//2),nn.SiLU(),nn.Dropout(dr),nn.Linear(h//2,1))
        if use_loc_head:
            s.loc_head=nn.Sequential(nn.Linear(h,h//2),nn.SiLU(),nn.Linear(h//2,1))

    def forward(s, d):
        h=s.embed(d.x); pos=d.pos; t=d.x[:,s.n_pre:]
        for l,f in zip(s.ls,s.fs): h,pos=l(h,pos,d.edge_index,d.edge_attr); h=f(h,t)
        logits = s.head(h).squeeze(-1)
        pred_center = None
        if s.use_loc_head:
            batch = d.batch if hasattr(d, "batch") and d.batch is not None \
                    else torch.zeros(h.size(0), dtype=torch.long, device=h.device)
            loc_logit = s.loc_head(h).squeeze(-1)
            attn = pyg_softmax(loc_logit, batch)                    # sums to 1 per graph
            weighted = attn.unsqueeze(-1) * pos                     # (N,3)
            n_graphs = int(batch.max().item()) + 1
            pred_center = torch.zeros(n_graphs, 3, device=h.device).index_add_(0, batch, weighted)
        return logits, pred_center

## Cell 6 — Training machinery (identical to Phase 14/20/53/56)

In [ ]:
def run_epoch(model, loader, opt=None, lam=0.0):
    tr = opt is not None; model.train(tr)
    crit = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([POS_WEIGHT], device=DEVICE))
    huber = nn.SmoothL1Loss(beta=4.0)                    # ~matches the 4A DCC threshold
    ps, ys, cd = [], [], []
    for b in loader:
        b = b.to(DEVICE)
        with torch.set_grad_enabled(tr):
            logits, pred_c = model(b)
            loss = crit(logits, b.y)
            if lam > 0 and pred_c is not None:
                true_c = b.true_center.view(-1, 3)
                loss = loss + lam * huber(pred_c, true_c)
                cd.append((pred_c - true_c).norm(dim=1).detach().cpu().numpy())
        if tr:
            opt.zero_grad(set_to_none=True); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        ps.append(torch.sigmoid(logits).detach().cpu().numpy()); ys.append(b.y.cpu().numpy())
    p, y = np.concatenate(ps), np.concatenate(ys)
    out = {"pr": average_precision_score(y, p), "roc": roc_auc_score(y, p)}
    if cd:
        d_ = np.concatenate(cd)
        out["dcc"] = float((d_ < 4.0).mean()); out["mean_center_err"] = float(d_.mean())
    return out


def fit(use_loc_head, lam, seed, epochs=40, patience=8, batch=16):
    torch.manual_seed(seed); np.random.seed(seed)
    model = LocalizingEGNN(N_PRE, TOPO_DIM, use_loc_head=use_loc_head).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    tl = DataLoader(splits["train"], batch_size=batch, shuffle=True)
    vl = DataLoader(splits["val"], batch_size=batch)
    best, state, bad = -1, None, 0
    for ep in range(epochs):
        run_epoch(model, tl, opt, lam)
        v = run_epoch(model, vl, lam=lam)
        sch.step()
        if v["pr"] > best:
            best, state, bad = v["pr"], {k:t.clone() for k,t in model.state_dict().items()}, 0
        else:
            bad += 1
            if bad >= patience: break
    model.load_state_dict(state); model.eval()
    return model, best

## Cell 7 — Build H0 / H1 / H2 / burial column masks (identical to Phase 23, applied to the round-3 topology block already loaded in Cell 1 as blob6b/TOPO_NAMES -- NOT reloading phase6b_graphs.pt, that would silently swap back to the original-scale topology block)

In [ ]:
BURIAL_NAMES = set(blob6b["burial_names"])   # blob6b/TOPO_NAMES already loaded in Cell 1 (phase51_graphs.pt)
print(f"topo columns: {len(TOPO_NAMES)} -- {TOPO_NAMES[:3]} ... {TOPO_NAMES[-2:]}")

MASKS = {
    "H0_only":     np.array([n.startswith("H0_") for n in TOPO_NAMES]),
    "H1_only":     np.array([n.startswith("H1_") for n in TOPO_NAMES]),
    "H2_only":     np.array([n.startswith("H2_") for n in TOPO_NAMES]),
    "burial_only": np.array([n in BURIAL_NAMES for n in TOPO_NAMES]),
}
for name, mask in MASKS.items():
    kept = [n for n,k in zip(TOPO_NAMES, mask) if k]
    print(f"  {name:<14} {mask.sum()} columns: {kept}")
assert sum(m.sum() for m in MASKS.values()) == len(TOPO_NAMES), \
    "masks should exactly partition the topo block (no overlap, no gaps)"
print("masks partition the full topo block exactly -- no overlap, no gaps")

## Cell 8 — Load Phase 56's already-trained 'full' arm (seeds 0-2, matched), train the 5 OTHER arms fresh at matched 3 seeds

In [ ]:
BEST_LAMBDA = 0.1
WORK = Path("/kaggle/working"); (WORK/"cache").mkdir(parents=True, exist_ok=True)

arm_models = {"full": {}}
for seed in (0,1,2):
    cands = []
    for r in (Path("/kaggle/input"), Path("/kaggle/working"), Path(".")):
        if r.exists(): cands.extend(r.rglob(f"clean_deploy_scaled3_seed{seed}.pt"))
    assert cands, f"clean_deploy_scaled3_seed{seed}.pt not found -- attach Phase 56's kernel output"
    state = torch.load(max(cands, key=lambda p: p.stat().st_size), map_location=DEVICE)
    m = LocalizingEGNN(N_PRE, TOPO_DIM).to(DEVICE); m.load_state_dict(state); m.eval()
    arm_models["full"][seed] = m
print(f"loaded {len(arm_models['full'])} 'full' checkpoints from Phase 56 (no retraining)")

ORIG_X = [d.x.clone() for d in dataset]
def apply_mask(mask):
    for d, x0 in zip(dataset, ORIG_X):
        if mask is None:
            d.x = torch.cat([x0[:, :N_PRE], torch.zeros_like(x0[:, N_PRE:])], dim=1)
        else:
            t = x0[:, N_PRE:].clone()
            keep = torch.from_numpy(mask)
            t[:, ~keep] = 0.0
            d.x = torch.cat([x0[:, :N_PRE], t], dim=1)
def restore():
    for d, x0 in zip(dataset, ORIG_X):
        d.x = x0.clone()

ARMS_TO_TRAIN = {"no_topo": None, **MASKS}   # None = zero the whole topo block
t0 = time.time()
for arm_name, mask in ARMS_TO_TRAIN.items():
    apply_mask(mask)
    arm_models[arm_name] = {}
    for seed in (0,1,2):
        m, val_pr = fit(use_loc_head=True, lam=BEST_LAMBDA, seed=seed)
        v = run_epoch(m, DataLoader(splits["val"], batch_size=16), lam=BEST_LAMBDA)
        arm_models[arm_name][seed] = m
        torch.save(m.state_dict(), WORK/f"cache/{arm_name}_scaled3_seed{seed}.pt")
        print(f"  [{arm_name}] seed {seed}: val PR {val_pr:.4f}  "
              f"val DCC {v.get('dcc',float('nan')):.4f}  [{(time.time()-t0)/60:.1f}m]")
    restore()

print(f"\ntrained {sum(len(v) for k,v in arm_models.items() if k!='full')} new models "
      f"across {len(ARMS_TO_TRAIN)} arms, reused 3 for 'full' -- all six arms now matched at 3 seeds")

## Cell 9 — Ground-truth ligands for ALL 420 COACH420 structures (identical to Phase 53/56)

In [ ]:
NON_LIGAND={"HOH","DOD","WAT","SO4","PO4","GOL","EDO","PEG","PG4","PGE","1PE","MPD","TRS",
            "ACT","ACY","FMT","CIT","MES","EPE","DMS","IMD","BME","NO3","AZI","CO3","IOD",
            "BR","CL","F","NA","K","CS","RB","LI","UNX","UNL","UNK"}
CATALYTIC_METALS={"ZN","MG","MN","FE","FE2","CU","CU1","CA","NI","CO","CD","W","MO"}
MIN_RES, MAX_RES, MIN_LIG_ATOMS = 50, 1000, 6

def parse_structure(pid, path, wanted_chain):
    try:
        with gzip.open(path, "rt") as fh:
            model = next(PDBParser(QUIET=True, PERMISSIVE=True).get_structure(pid, fh).get_models())
    except Exception:
        return []
    ligs = []
    for ch in model:
        for res in ch:
            if res.id[0] == " ": continue
            nm = res.get_resname().strip().upper()
            if nm in AA3_TO_1 or nm in NON_LIGAND: continue
            xyz = np.asarray([a.coord for a in res.get_unpacked_list() if a.element not in ("H","D")],
                            dtype=np.float32).reshape(-1,3)
            is_metal = nm in CATALYTIC_METALS
            if is_metal or len(xyz) >= MIN_LIG_ATOMS: ligs.append(xyz)
    out = []
    chains = [model[wanted_chain]] if (wanted_chain and wanted_chain in [c.id for c in model]) \
             else [c for c in model]
    for ch in chains:
        residues=[r for r in ch if r.get_resname().strip().upper() in AA3_TO_1 and "CA" in r]
        if not (MIN_RES <= len(residues) <= MAX_RES): continue
        ca=[r["CA"].coord for r in residues]
        seq="".join(AA3_TO_1.get(r.get_resname().strip().upper(),"X") for r in residues)
        atoms=[np.asarray([a.coord for a in r.get_unpacked_list() if a.element not in ("H","D")],
                          dtype=np.float32).reshape(-1,3) for r in residues]
        out.append((ch.id, seq, np.asarray(ca,dtype=np.float32), atoms, ligs))
    return out

t0=time.time()
eval_units = []
for pid, ch in coach:
    if pid not in coach_paths: continue
    chains = parse_structure(pid, coach_paths[pid], ch)
    if chains: eval_units.append((pid, chains[:1]))   # coach420: one specific target chain
print(f"evaluable COACH420 structures: {len(eval_units)}/{len(coach)}  ({(time.time()-t0)/60:.1f} min)")

## Cell 10 — Featurize (identical pipeline: dihedral + ESM2 + topology, matched to the round-3 topology block; cache filename updated)

In [ ]:
import gudhi
from transformers import AutoTokenizer, AutoModel
from Bio.PDB import PPBuilder

AA_ALPHABET="ACDEFGHIKLMNPQRSTVWYX"; A2I={a:i for i,a in enumerate(AA_ALPHABET)}
MAX_HOM_DIM=2; BETTI_RADII=(2.0,3.0,5.0); NBHD_RADIUS=12.0; MAX_NBHD_ATOMS=300
FPD=6+2+len(BETTI_RADII)

_FULL_TOPO_NAMES = [f"H{d}_{s}" for d in range(MAX_HOM_DIM+1)
                    for s in ["n_bars","total_pers","max_pers","mean_pers","std_pers","entropy",
                              "mean_birth","mean_death", *[f"betti_r{r:g}" for r in BETTI_RADII]]] \
                   + ["nbhd_n_atoms","nbhd_gyration"]
_TOPO_KEEP = np.array([n in set(TOPO_NAMES) for n in _FULL_TOPO_NAMES])
assert _TOPO_KEEP.sum() == TOPO_DIM

def rgc(pos,r=8.0,k=32):
    n=pos.size(0); pos=pos-pos.mean(0,keepdim=True)
    d=torch.cdist(pos,pos).clone(); d.fill_diagonal_(float("inf")); m=d<=r
    if k<n:
        _,i=torch.topk(d,k,dim=1,largest=False); keep=torch.zeros_like(m); keep.scatter_(1,i,True); m&=keep
    m=m|m.t(); t,s=m.nonzero(as_tuple=True); return torch.stack([s,t]).long()

def _dg(pts):
    pts=np.ascontiguousarray(pts,dtype=np.float64); st=None; sq=True
    if len(pts)>=5:
        try: st=gudhi.AlphaComplex(points=pts).create_simplex_tree()
        except Exception: st=None
    if st is None:
        st=gudhi.RipsComplex(points=pts,max_edge_length=float(NBHD_RADIUS)).create_simplex_tree(max_dimension=MAX_HOM_DIM+1); sq=False
    st.compute_persistence(homology_coeff_field=2,persistence_dim_max=True)
    o={}
    for d in range(MAX_HOM_DIM+1):
        b=np.asarray(st.persistence_intervals_in_dimension(d),dtype=np.float64).reshape(-1,2)
        o[d]=np.sqrt(np.maximum(b,0.0)) if (sq and len(b)) else b
    return o

def _vec(dg,pts):
    f=[]
    for d in range(MAX_HOM_DIM+1):
        bars=dg.get(d,np.zeros((0,2)))
        fin=bars[np.isfinite(bars).all(axis=1)] if len(bars) else bars.reshape(-1,2)
        if len(fin)==0: f.extend([0.0]*FPD); continue
        b,dt=fin[:,0],fin[:,1]; pers=np.maximum(dt-b,0.0); tot=float(pers.sum())
        ent=0.0
        if tot>0:
            p=pers/tot; nz=p[p>0]; ent=float(-(nz*np.log(nz)).sum())
        f.extend([float(len(fin)),tot,float(pers.max()),float(pers.mean()),float(pers.std()),
                  ent,float(b.mean()),float(dt.mean()),
                  *[float(((b<=r)&(dt>r)).sum()) for r in BETTI_RADII]])
    c=pts-pts.mean(0,keepdims=True)
    f.extend([float(len(pts)),float(np.sqrt((c**2).sum(1).mean())) if len(pts) else 0.0])
    return np.asarray(f,dtype=np.float32)

def topo_chain(ca,per_res):
    atoms=np.concatenate([a for a in per_res if len(a)],axis=0)
    tree=cKDTree(atoms); out=np.zeros((len(ca),TOPO_DIM),np.float32)
    for i,c in enumerate(ca):
        idx=tree.query_ball_point(c,NBHD_RADIUS)
        if len(idx)>MAX_NBHD_ATOMS: _,idx=tree.query(c,k=MAX_NBHD_ATOMS)
        pts=atoms[np.asarray(idx,dtype=int)]
        if len(pts)>=4: out[i]=_vec(_dg(pts),pts)[_TOPO_KEEP]
    return out

def dihedrals(pdb_id, chain_id, path, n_expect):
    try:
        with gzip.open(path,"rt") as fh:
            model=next(PDBParser(QUIET=True,PERMISSIVE=True).get_structure(pdb_id,fh).get_models())
        ch=model[chain_id]
    except Exception: return None
    residues=[r for r in ch if r.get_resname().strip().upper() in AA3_TO_1 and "CA" in r]
    if len(residues)!=n_expect: return None
    ppmap={}
    for pp in PPBuilder().build_peptides(ch, aa_only=False):
        for res,(phi,psi) in zip(pp, pp.get_phi_psi_list()): ppmap[res.id]=(phi,psi)
    phis,psis=[],[]
    for r in residues:
        phi,psi=ppmap.get(r.id,(None,None)); phis.append(phi); psis.append(psi)
    def enc(a):
        v=torch.tensor([x if x is not None else 0.0 for x in a],dtype=torch.float32)
        m=torch.tensor([x is not None for x in a],dtype=torch.float32)
        return torch.stack([torch.sin(v)*m,torch.cos(v)*m],1), m.unsqueeze(1)
    pf,pm=enc(phis); sf,sm=enc(psis)
    return torch.cat([pf,sf,pm,sm],1)

tok = AutoTokenizer.from_pretrained("facebook/esm2_t12_35M_UR50D")
esm = AutoModel.from_pretrained("facebook/esm2_t12_35M_UR50D").to(DEVICE).eval()

@torch.no_grad()
def embed_seq(seq):
    enc = tok([seq], return_tensors="pt", truncation=True, max_length=1002)
    enc = {k:v.to(DEVICE) for k,v in enc.items()}
    out = esm(**enc).last_hidden_state
    return out[0,1:1+len(seq),:].float().cpu()

t0=time.time(); built = {}
n_total = sum(len(c) for _,c in eval_units)
done=0
for pid, chains in eval_units:
    for cid, seq, ca, per_res, ligs in chains:
        n = len(ca)
        oh = torch.zeros(n, 21); oh[torch.arange(n), torch.tensor([A2I.get(c,20) for c in seq])]=1.0
        dih = dihedrals(pid, cid, coach_paths[pid], n)
        if dih is None: continue
        esm_e = embed_seq(seq)
        topo = topo_chain(ca, per_res)
        pos = torch.from_numpy(ca); pos = pos - pos.mean(0, keepdim=True)
        x = torch.cat([oh, dih, esm_e, torch.from_numpy(topo.astype(np.float32))], dim=1)
        ei = rgc(pos)
        d_ = (pos[ei[0]]-pos[ei[1]]).norm(dim=1,keepdim=True)
        sep = (ei[0]-ei[1]).abs().float().unsqueeze(1)
        built[(pid,cid)] = dict(x=x, pos=pos, edge_index=ei,
                                edge_attr=torch.cat([d_,torch.log1p(sep)],1), ca=ca, ligs=ligs)
        done += 1
    if done % 40 < 1: print(f"  {done}/{n_total}  [{(time.time()-t0)/60:.1f}m]")

del esm; torch.cuda.empty_cache()
print(f"\nfeaturized {len(built)}/{n_total} chains in {(time.time()-t0)/60:.1f} min")

raw_cache = {k: {kk: (vv.numpy() if torch.is_tensor(vv) else vv) for kk,vv in v.items()}
            for k,v in built.items()}
with open(WORK/"cache/phase58_raw_features.pkl", "wb") as fh:
    pickle.dump(raw_cache, fh)
print(f"cached raw features -> {(WORK/'cache/phase58_raw_features.pkl').stat().st_size/1e6:.0f} MB")

## Cell 11 — P2Rank on the FULL 420 (same-run controlled reference, identical to Phase 53/56)

In [ ]:
t0=time.time(); PRANK=None
subprocess.run("curl -sL https://github.com/rdk/p2rank/releases/download/2.4.2/p2rank_2.4.2.tar.gz "
               "-o /tmp/p2.tar.gz && tar xzf /tmp/p2.tar.gz -C /tmp", shell=True, check=True, timeout=1200)
PRANK = str(next(Path("/tmp").glob("p2rank_*/prank"))); os.chmod(PRANK, 0o755)

pW = Path("/tmp/p2_53"); shutil.rmtree(pW, ignore_errors=True); pW.mkdir()
pOUT = Path("/tmp/p2_53_out"); shutil.rmtree(pOUT, ignore_errors=True)
for pid in sorted(coach_paths):
    plain = pW / f"{pid}.pdb"
    if not plain.exists():
        with gzip.open(coach_paths[pid],"rb") as fi, open(plain,"wb") as fo: shutil.copyfileobj(fi,fo)
(pW/"ds.ds").write_text("\n".join(str(pW/f"{p}.pdb") for p in sorted(coach_paths))+"\n")

r = subprocess.run([PRANK,"predict",str(pW/"ds.ds"),"-o",str(pOUT),
                    "-threads", str(max(1, os.cpu_count() or 2))],
                   capture_output=True, text=True, timeout=3600)
print(f"P2Rank rc={r.returncode} in {(time.time()-t0)/60:.1f} min")
pred_csvs = {p.stem.replace(".pdb_predictions",""): p for p in pOUT.rglob("*_predictions.csv")}
print(f"P2Rank outputs: {len(pred_csvs)}/{len(coach_paths)}")

def p2rank_top_center(pid):
    f = pred_csvs.get(pid)
    if f is None: return None
    with open(f) as fh:
        rows = list(_csv.DictReader(fh))
    if not rows: return None
    rows = [{k.strip(): v.strip() if isinstance(v,str) else v for k,v in row.items() if k} for row in rows]
    cx = next((k for k in rows[0] if k.lower().replace("_","")=="centerx"), None)
    cy = next((k for k in rows[0] if k.lower().replace("_","")=="centery"), None)
    cz = next((k for k in rows[0] if k.lower().replace("_","")=="centerz"), None)
    if not (cx and cy and cz): return None
    try:
        return np.array([float(rows[0][cx]), float(rows[0][cy]), float(rows[0][cz])], dtype=np.float32)
    except (ValueError, KeyError):
        return None

p2rank_centers = {pid: p2rank_top_center(pid) for pid in coach_paths}
print(f"P2Rank centers parsed: {sum(1 for v in p2rank_centers.values() if v is not None)}/{len(coach_paths)}")

## Cell 12 — Score all six arms (matched 3 seeds each), per-seed pairwise significance table

In [ ]:
mu, sd = blob6b["topo_mean"], blob6b["topo_std"]

@torch.no_grad()
def predict_chain(entry, model_dict):
    x = entry["x"].clone()
    x[:, N_PRE:] = (x[:, N_PRE:] - torch.from_numpy(mu)) / torch.from_numpy(sd)
    d = type("D", (), {})()
    d.x=x.to(DEVICE); d.pos=entry["pos"].to(DEVICE)
    d.edge_index=entry["edge_index"].to(DEVICE); d.edge_attr=entry["edge_attr"].to(DEVICE)
    d.batch = torch.zeros(x.size(0), dtype=torch.long, device=DEVICE)
    seed_logits, seed_centers = [], []
    for m in model_dict.values():
        lg, pc = m(d)
        seed_logits.append(torch.sigmoid(lg).cpu().numpy())
        seed_centers.append(pc.cpu().numpy()[0])
    offset = entry["ca"].mean(axis=0)
    return float(np.mean(seed_logits, axis=0).sum()), np.mean(seed_centers, axis=0) + offset


def dcc(pred_center, ligs):
    if pred_center is None or not ligs: return None
    best = np.inf
    for lig_atoms in ligs:
        if len(lig_atoms) == 0: continue
        best = min(best, float(np.linalg.norm(pred_center - lig_atoms.mean(axis=0))))
    return best if np.isfinite(best) else None


by_structure = defaultdict(list)
for (pid, cid), entry in built.items():
    by_structure[pid].append((cid, entry))

def score_arm(model_dict):
    hits, n = 0, 0
    for pid, chain_list in by_structure.items():
        ligs = chain_list[0][1]["ligs"]
        best_conf, best_center = -np.inf, None
        for cid, entry in chain_list:
            conf, center = predict_chain(entry, model_dict)
            if conf > best_conf: best_conf, best_center = conf, center
        d_ = dcc(best_center, ligs)
        if d_ is None: continue
        n += 1; hits += int(d_ < 4.0)
    return hits / max(n, 1), n

print(f"{'arm':<14}{'DCC (3-seed matched)':>22}{'n':>6}")
print("-"*42)
summary = {}
for arm_name, model_dict in arm_models.items():
    rate, n = score_arm(model_dict)
    summary[arm_name] = {"dcc": rate, "n": n}
    print(f"{arm_name:<14}{rate:>22.4f}{n:>6}")

base = summary["no_topo"]["dcc"]; full = summary["full"]["dcc"]; total_gain = full - base
print(f"\nno_topo baseline: {base:.4f}  full: {full:.4f}  (total gain: {total_gain:+.4f})")

decomp_arms = ["H0_only", "H1_only", "H2_only", "burial_only"]
ranked = sorted(decomp_arms, key=lambda n: -summary[n]["dcc"])
print(f"\n{'arm':<14}{'DCC':>10}{'gain over no_topo':>20}{'% of full gain':>18}")
print("-"*62)
for name in ranked:
    g = summary[name]["dcc"] - base
    pct = 100 * g / total_gain if total_gain > 1e-9 else float("nan")
    print(f"{name:<14}{summary[name]['dcc']:>10.4f}{g:>+20.4f}{pct:>17.1f}%")
print(f"\nmatched 3-seed ranking: {' > '.join(ranked)}")

## Cell 13 — Per-seed pairwise significance table across all four decomposition arms

In [ ]:
def per_seed_dcc(arm_name):
    rates = []
    for seed, m in sorted(arm_models[arm_name].items()):
        rate, n = score_arm({seed: m})
        rates.append(rate)
    return np.array(rates)

per_seed = {name: per_seed_dcc(name) for name in ["full", "no_topo"] + decomp_arms}
for name in per_seed:
    r = per_seed[name]
    print(f"{name:<14} per-seed {np.round(r,4).tolist()}  mean {r.mean():.4f}  std {r.std():.4f}")

from itertools import combinations
print(f"\n{'pair':<28}{'mean diff':>12}{'noise (max std)':>18}{'wins/3':>10}{'diff/noise':>12}")
print("-"*82)
pairwise = {}
for a, b in combinations(decomp_arms, 2):
    ra, rb = per_seed[a], per_seed[b]
    diff = ra.mean() - rb.mean()
    noise = max(ra.std(), rb.std())
    wins = int((ra > rb).sum())
    ratio = abs(diff) / noise if noise > 1e-9 else float("inf")
    pairwise[f"{a}_vs_{b}"] = {"diff": float(diff), "noise": float(noise), "wins": wins, "ratio": float(ratio)}
    print(f"{a+' vs '+b:<28}{diff:>+12.4f}{noise:>18.4f}{wins:>7}/3{ratio:>12.2f}x")

verdict = ("ATTRIBUTABLE -- at least one pair clears 1.5x noise" if
           max(v["ratio"] for v in pairwise.values()) > 1.5 else
           "STILL DIFFUSE -- no pair clears 1.5x noise, matches the original-scale finding")
print(f"\nVERDICT: {verdict}")

json.dump({"summary": summary, "per_seed": {k: v.tolist() for k,v in per_seed.items()},
           "pairwise": pairwise, "ranked": ranked, "verdict": verdict},
          open(WORK/"phase58_summary.json", "w"), indent=2)
print("\nsaved -> phase58_summary.json")

---

## Reading this

- **Still diffuse (no pair clears 1.5x noise)** -- confirms the original
  finding is scale-independent: this decomposition genuinely cannot
  attribute the topology benefit to one homology dimension, with 3.65x
  more data. A real, stable negative result.
- **One or more pairs now separate clearly** -- more data sharpened a
  signal that was too noisy to see at the original scale; worth an 8-seed
  confirmation before fully trusting which arm(s) actually separated.